**MGMT298D: Science and Strategy of AI**
# Week 4: Neural Networks

#### This notebook introduces neural networks for classification using `keras` (via `tensorflow`). We build progressively deeper networks on the breast cancer dataset and use EarlyStopping to avoid overfitting.

# 1 Setup & Data

#### We import `numpy`, `pandas`, `sklearn` for data loading and preprocessing, and `tensorflow.keras` for building neural networks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# 1.1 Load Data

#### We load the Wisconsin Breast Cancer dataset from `sklearn.datasets`. It has 30 numeric features and a binary target (benign vs malignant).

In [ ]:
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Class distribution: {y.value_counts().to_dict()}")
X.head()

# 1.2 Train/Test Split & Scaling

#### We split 80/20 with stratification to preserve class balance, then standardize features with `StandardScaler` — important for neural networks to train well.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape[0]} samples")
print(f"Test: {X_test_scaled.shape[0]} samples")
print(f"Features: {X_train_scaled.shape[1]}")

---
# 2 Simple Neural Network

#### We start with the simplest possible network — one hidden layer with 4 neurons and relu activation, followed by a sigmoid output for binary classification.

In [ ]:
model_simple = Sequential([
    Dense(4, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(1, activation='sigmoid')
])

model_simple.compile(optimizer='adam', loss='binary_crossentropy')

history_simple = model_simple.fit(X_train_scaled, y_train, epochs=50,
                                   validation_split=0.15)

y_pred_simple = (model_simple.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
acc_simple = accuracy_score(y_test, y_pred_simple)
print(f"Simple NN Test Accuracy: {acc_simple:.4f}")

In [ ]:
plt.plot(history_simple.history['loss'], label='Train')
plt.plot(history_simple.history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Simple NN Training')
plt.legend()
plt.show()

---
# 3 Deep Neural Network

#### Adding more layers (64 → 32 → 16) lets the network learn more complex representations, but without regularization it may overfit on a small dataset like this.

In [ ]:
model_deep = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_deep.compile(optimizer='adam', loss='binary_crossentropy')

history_deep = model_deep.fit(X_train_scaled, y_train, epochs=50,
                               validation_split=0.15, verbose=0)

y_pred_deep = (model_deep.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
acc_deep = accuracy_score(y_test, y_pred_deep)
print(f"Deep NN Test Accuracy: {acc_deep:.4f}")

In [ ]:
plt.plot(history_deep.history['loss'], label='Train')
plt.plot(history_deep.history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Deep NN Training')
plt.legend()
plt.show()

---
# 4 Regularized Neural Network

#### We train the same deep architecture but add `EarlyStopping`, which halts training when validation loss stops improving and restores the best weights. This is a simple, effective way to prevent overfitting.

In [ ]:
model_reg = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_reg.compile(optimizer='adam', loss='binary_crossentropy')

early_stop = EarlyStopping(patience=15, restore_best_weights=True)

history_reg = model_reg.fit(X_train_scaled, y_train, epochs=75,
                             validation_split=0.15, callbacks=[early_stop], verbose=0)

y_pred_reg = (model_reg.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
acc_reg = accuracy_score(y_test, y_pred_reg)
print(f"Regularized NN Test Accuracy: {acc_reg:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_reg))

In [ ]:
plt.plot(history_reg.history['loss'], label='Train')
plt.plot(history_reg.history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Regularized NN Training')
plt.legend()
plt.show()

---
# 5 Model Comparison

#### Final side-by-side comparison of all three models on test accuracy.

In [ ]:
print(f"Simple NN:       {acc_simple:.4f}")
print(f"Deep NN:         {acc_deep:.4f}")
print(f"Regularized NN:  {acc_reg:.4f}")

In [ ]:
models = ['Simple NN', 'Deep NN', 'Regularized NN']
accs = [acc_simple, acc_deep, acc_reg]
plt.bar(models, accs)
plt.ylabel('Test Accuracy')
plt.title('Model Comparison')
plt.ylim(0.9, 1.0)
plt.show()

---
# 6 Effect of Hidden Layer Size

#### We sweep the hidden layer size from 8 to 128 in a single-layer network to see how capacity affects accuracy.

In [ ]:
hidden_sizes = [8, 16, 32, 64, 128]
accuracies_by_size = []

for size in hidden_sizes:
    model_size = Sequential([
        Dense(size, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dense(1, activation='sigmoid')
    ])
    model_size.compile(optimizer='adam', loss='binary_crossentropy')
    model_size.fit(X_train_scaled, y_train, epochs=50, validation_split=0.15, verbose=0)

    y_pred_size = (model_size.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
    acc_size = accuracy_score(y_test, y_pred_size)
    accuracies_by_size.append(acc_size)

print("Hidden Size vs Accuracy:")
for size, acc in zip(hidden_sizes, accuracies_by_size):
    print(f"  Size {size:3d}: {acc:.4f}")

In [ ]:
plt.plot(hidden_sizes, accuracies_by_size, 'o-')
plt.xlabel('Hidden Layer Size')
plt.ylabel('Test Accuracy')
plt.title('Effect of Hidden Layer Size')
plt.show()